In [ ]:
''' import os, sys
# ensure the project root is on sys.path
sys.path.append(os.path.abspath(".."))

import pandas as pd
from src.data_loader import load_lyrics_in_chunks, filter_english, clean_lyrics_column, sample_and_save

DATA_PATH = "../data/song_lyrics.csv"    # adjust if your data folder differs
SAMPLE_OUT = "../data/sample_100k.csv"
CHUNKSIZE = 50_000
USECOLS = ["tag", "language", "lyrics"]

reader = load_lyrics_in_chunks(DATA_PATH, usecols=USECOLS, chunksize=CHUNKSIZE)
df_chunk = next(reader)
print("Raw chunk shape:", df_chunk.shape)
df_chunk.head(3)

df_en = filter_english(df_chunk)
print("After English filter:", df_en.shape)
df_en["tag"].value_counts().head(5)

df_clean = clean_lyrics_column(df_en)
# show original vs cleaned for a few rows
df_clean[["lyrics", "clean_lyrics"]].sample(3, random_state=40)

sample_and_save(df_clean, SAMPLE_OUT, n=100_000)
print("Sample saved to", SAMPLE_OUT) '''

In [3]:
# ─── Full‐dataset load, clean, then stratified sample ───

import os, sys
# ensure the project root is on sys.path
sys.path.append(os.path.abspath(".."))

import pandas as pd
from src.data_loader import load_lyrics_in_chunks, filter_english, clean_lyrics_column

DATA_PATH = "../data/song_lyrics.csv"
CHUNKSIZE = 50_000

# 1) Iterate through every chunk, filter & clean
reader = load_lyrics_in_chunks(
    DATA_PATH,
    usecols=["tag", "language", "lyrics"],
    chunksize=CHUNKSIZE
)

all_chunks = []
for chunk in reader:
    df_en    = filter_english(chunk)
    df_clean = clean_lyrics_column(df_en)
    all_chunks.append(df_clean)

df_all = pd.concat(all_chunks, ignore_index=True)
print("After full load & clean:", df_all.shape)
print(df_all.tag.value_counts())

# 2a) Stratified sample: up to 5000 per genre
sample = (
    df_all
    .groupby("tag", group_keys=False)
    .apply(lambda g: g.sample(min(len(g), 5000), random_state=42))
)

# 2b) (Alternatively, an overall 100k-sample)
# sample = df_all.sample(100_000, random_state=42)

# 3) Persist
sample.to_csv("../data/sample_stratified.csv", index=False)
print("Stratified sample saved:", sample.shape)
print(sample.tag.value_counts())

After full load & clean: (3374198, 4)
tag
pop        1393559
rap         964605
rock        633308
rb          155082
misc        140986
country      86658
Name: count, dtype: int64


/var/folders/5c/ndh3_tdn32j2s6jw_0w3qchm0000gn/T/ipykernel_73933/2228231752.py:32: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_all


Stratified sample saved: (30000, 4)
tag
country    5000
misc       5000
pop        5000
rap        5000
rb         5000
rock       5000
Name: count, dtype: int64
